# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR^2 dataset using the `mlcroissant` library. All entities, such as record sets and fields, are referenced by their `@id` for precise identification.

### Dataset Source
The dataset is described using a Croissant schema accessible at the following URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL for FAIR^2
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, columns, and their `@id`s.

**This step will enumerate the record sets and their fields from the dataset, referencing them consistently by their `@id`.**

In [ ]:
# List all record sets by @id and their fields
print("RecordSets and fields available in the dataset:\n")
for rs in dataset.metadata.recordSets:
    print(f"RecordSet @id: {rs.id} | name: {rs.name if hasattr(rs, 'name') else ''}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - Field @id: {field.id} | name: {field.name if hasattr(field, 'name') else ''} | datatype: {getattr(field, 'dataType', '')}")
    print()

# Save first record set id for further analysis
record_set_ids = [rs.id for rs in dataset.metadata.recordSets]
# If the dataset contains no recordSets, this will be empty

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. All operations are based on the record set and field `@id`s as explored above.

In [ ]:
# Extract data from all main record sets into a dict of DataFrames keyed by record set @id
print(f"Extracting data for the following record sets: {record_set_ids}")
dataframes = {}

for rsid in record_set_ids:
    records = list(dataset.records(record_set=rsid))
    if records:
        df = pd.DataFrame(records)
        dataframes[rsid] = df
        print(f"Loaded DataFrame for RecordSet {rsid} with {len(df)} records.")
    else:
        print(f"No records loaded for RecordSet {rsid}.")

# Show columns for the first available (non-empty) DataFrame
sample_rs_id = None
for k, v in dataframes.items():
    if len(v) > 0:
        print(f"\nColumns in RecordSet {k} DataFrame: {v.columns.tolist()}")
        sample_rs_id = k
        break
if sample_rs_id:
    display(dataframes[sample_rs_id].head())
else:
    print("No data found in any record sets.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps—such as filtering, normalization, and grouping—referencing fields strictly by their `@id`. Adjust field IDs below to match your dataset's fields.

In [ ]:
if sample_rs_id and not dataframes[sample_rs_id].empty:
    df = dataframes[sample_rs_id]
    # Try to find a numeric field automatically
    numeric_fields = df.select_dtypes(include=['number']).columns.tolist()
    if numeric_fields:
        numeric_field = numeric_fields[0]
        print(f"Selected numeric field for EDA: {numeric_field}")

        # Filter for values above a threshold (arbitrary, e.g., mean)
        threshold = df[numeric_field].mean()
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records where {numeric_field} > {threshold:.2f} (mean): {len(filtered_df)} rows")
        display(filtered_df.head())

        # Normalize the numeric field
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} (z-score):")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try to group by the first non-numeric field if available
        group_fields = [col for col in df.columns if col != numeric_field and df[col].dtype == object]
        if group_fields:
            group_field = group_fields[0]
            print(f"Grouping by {group_field}")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            display(grouped_df.head())
        else:
            print("No suitable categorical field found to group by.")
    else:
        print("No numeric fields found in this record set.")
else:
    print("No suitable data available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below, we plot the distribution of the chosen numeric field and (if appropriate) boxplots/grouped averages.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if sample_rs_id and not dataframes[sample_rs_id].empty and 'numeric_field' in locals():
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field} in RecordSet {sample_rs_id}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    # If we computed a grouping above, show a barplot
    if 'grouped_df' in locals():
        plt.figure(figsize=(10, 4))
        sns.barplot(data=grouped_df, x=group_field, y=numeric_field)
        plt.title(f"Mean of {numeric_field} grouped by {group_field}")
        plt.ylabel(f"Mean {numeric_field}")
        plt.xlabel(group_field)
        plt.xticks(rotation=45)
        plt.show()
else:
    print("Not enough data to plot.")

## 6. Conclusion
In this notebook, we demonstrated how to load and explore the FAIR^2 dataset with `mlcroissant`, enumerate record sets and their fields by `@id`, load tabular data, run basic data analysis, and visualize results. For deeper domain-specific insights, refer to the provided metadata and the Croissant schema. All processing referenced the official Croissant `@id` fields for clarity and reproducibility.